In [15]:
%load_ext autoreload
%autoreload 2
import os
import csv
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from file_loader import get_all_files

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
import random
def set_seed(seed): 
    torch.backends.cudnn.deterministic = True 
    torch.backends.cudnn.benchmark = False 
    torch.manual_seed(seed) 
    torch.cuda.manual_seed_all(seed) 
    np.random.seed(seed) 
    random.seed(seed)
    
set_seed(42)

In [64]:
data = 'data/good_images/'                                # total DIC labelled crops = 12825
files = get_all_files(data, 'png')                   # 12825-191(not proper) = 12634                                                
good_image_names =[]                                      # not proper =  border crops, CDB_Sample125( BF crops)
for file in files:                                   # from 12634, we have 11071 good images(taken from BF)
    out_dir, img_name = os.path.split(file)
    _, sample_folder = os.path.split(out_dir)
    img_name = img_name.replace('JAI','jai')              # DIC image crop is with 'JAI'
    good_image_names.append(sample_folder+'/'+img_name)

In [65]:
len(good_image_names)

11071

In [67]:
data_df = pd.read_csv('data/all_data_new.csv')
clean_df = data_df.loc[data_df['image_name'].isin(good_image_names)].reset_index(drop = True)  #removing badly segmented crops
print(len(clean_df))
filt = (clean_df['artifact']!='ART')
clean_df = clean_df.loc[filt].reset_index(drop = True)           # removing artifacts (156)
clean_df = clean_df.drop(columns =['artifact'])
clean_df['hemo_dist'].replace({'HYO1':'HYPO', 'HYO2':'HYPO', 'HYO3':'HYPO', 'HYO4':'HYPO', \
                               'HYR1':'HYPR', 'HYR2':'HYPR', 'HYR3':'HYPR', 'HYR4':'HYPR'}, inplace=True)
clean_df.to_csv('data/all_data_new_clean1.csv', index=False)
print(len(clean_df))

11071
10915


#### Data which was labelled

In [68]:
data_df = pd.read_csv('data/all_data_new.csv')
print(data_df['size'].value_counts())
print("----------------------------------")
print(data_df['shape'].value_counts())
print("----------------------------------")
print(data_df['hemo_dist'].value_counts())
print("----------------------------------")
print(data_df['inclusion'].value_counts())
print("----------------------------------")
print(data_df['artifact'].value_counts())

NORM    11709
MICR     1309
NONE      461
MACR      250
Name: size, dtype: int64
----------------------------------
NONE    11102
ECHI      770
OVAL      632
TEAR      484
SCHI      169
ACAN      150
HELM      142
ELLI      118
SICK       83
SPHE       64
BITE       15
Name: shape, dtype: int64
----------------------------------
NONE    11469
HYO1      500
TARG      451
HYR1      370
HYO3      278
HYO2      260
STOM      140
HYO4      137
HYR2       52
HYR3       40
HYR4       32
Name: hemo_dist, dtype: int64
----------------------------------
NONE                12558
RETI                  401
MALA                  302
HOJO                  139
BAST, RETI            108
NRBC                   87
PABO                   65
BAST                   45
BAST, NRBC             10
BAST, HOJO              4
BAST, HOJO, RETI        4
HOJO, RETI              3
PABO, RETI              2
HOJO, NRBC              1
Name: inclusion, dtype: int64
----------------------------------
NONE    13268
ART    

#### Data free from bad images and artifacts

In [69]:
data_df_clean = pd.read_csv('data/all_data_new_clean1.csv')
print(data_df_clean['size'].value_counts())
print("----------------------------------")
print(data_df_clean['shape'].value_counts())
print("----------------------------------")
print(data_df_clean['hemo_dist'].value_counts())
print("----------------------------------")
print(data_df_clean['inclusion'].value_counts())

NORM    9617
MICR    1119
MACR     179
Name: size, dtype: int64
----------------------------------
NONE    8631
ECHI     696
OVAL     572
TEAR     423
SCHI     140
HELM     126
ACAN     120
ELLI     105
SPHE      54
SICK      37
BITE      11
Name: shape, dtype: int64
----------------------------------
NONE    9021
HYPO     995
HYPR     429
TARG     375
STOM      95
Name: hemo_dist, dtype: int64
----------------------------------
NONE                10261
RETI                  256
MALA                  151
HOJO                   87
BAST, RETI             65
BAST                   32
PABO                   30
NRBC                   23
HOJO, RETI              3
BAST, HOJO              3
BAST, HOJO, RETI        3
PABO, RETI              1
Name: inclusion, dtype: int64


#### Splitting data to train and test csv files and distributions of train labels

In [72]:
data_df_clean = pd.read_csv('data/all_data_new_clean1.csv')
train_inds, test_inds = train_test_split(np.array(list(range(data_df_clean.shape[0]))), test_size=0.2, random_state=2)
train_df = data_df_clean.iloc[train_inds,:].reset_index(drop=True)
test_df = data_df_clean.iloc[test_inds,:].reset_index(drop=True)

train_df.to_csv('data/train_new_clean.csv', index=False)
test_df.to_csv('data/test_new_clean.csv', index =False)

print(train_df['size'].value_counts())
print("----------------------------------")
print(train_df['shape'].value_counts())
print("----------------------------------")
print(train_df['hemo_dist'].value_counts())
print("----------------------------------")
print(train_df['inclusion'].value_counts())

NORM    7690
MICR     893
MACR     149
Name: size, dtype: int64
----------------------------------
NONE    6912
ECHI     570
OVAL     447
TEAR     347
SCHI     105
HELM      96
ACAN      95
ELLI      80
SPHE      41
SICK      30
BITE       9
Name: shape, dtype: int64
----------------------------------
NONE    7199
HYPO     809
HYPR     355
TARG     297
STOM      72
Name: hemo_dist, dtype: int64
----------------------------------
NONE                8208
RETI                 202
MALA                 121
HOJO                  74
BAST, RETI            55
PABO                  26
BAST                  21
NRBC                  18
BAST, HOJO, RETI       3
BAST, HOJO             2
HOJO, RETI             1
PABO, RETI             1
Name: inclusion, dtype: int64


##### Undersampling 50% of NORMAL crops    

In [74]:
norm_samples = np.where((train_df['size']=='NORM')&(train_df['shape']=='NONE')&(train_df['hemo_dist']=='NONE')&(train_df['inclusion']=='NONE'))[0]
#print(len(norm_samples))
drop_size = int(0.5*len(norm_samples))
drop_ind = np.random.choice(norm_samples, size = drop_size, replace = False)
train_df_undersampled = train_df.drop(train_df.index[list(drop_ind)])
train_df_undersampled = train_df_undersampled.sample(frac=1).reset_index(drop=True)

In [75]:
print(train_df_undersampled['size'].value_counts())
print("----------------------------------")
print(train_df_undersampled['shape'].value_counts())
print("----------------------------------")
print(train_df_undersampled['hemo_dist'].value_counts())
print("----------------------------------")
print(train_df_undersampled['inclusion'].value_counts())

NORM    5510
MICR     893
MACR     149
Name: size, dtype: int64
----------------------------------
NONE    4732
ECHI     570
OVAL     447
TEAR     347
SCHI     105
HELM      96
ACAN      95
ELLI      80
SPHE      41
SICK      30
BITE       9
Name: shape, dtype: int64
----------------------------------
NONE    5019
HYPO     809
HYPR     355
TARG     297
STOM      72
Name: hemo_dist, dtype: int64
----------------------------------
NONE                6028
RETI                 202
MALA                 121
HOJO                  74
BAST, RETI            55
PABO                  26
BAST                  21
NRBC                  18
BAST, HOJO, RETI       3
BAST, HOJO             2
HOJO, RETI             1
PABO, RETI             1
Name: inclusion, dtype: int64


In [76]:
df_copy = train_df_undersampled
min_samples = 200

size_rare_list = ['MACR']

for item in size_rare_list:
    size_samples = np.where((df_copy['size']==item))[0]
    len_samples = len(df_copy[df_copy['size']==item])
    gap_num = min_samples - len_samples
    temp_df = df_copy.iloc[np.random.choice(size_samples, size = gap_num)]
    df_copy = df_copy.append(temp_df, ignore_index = True)


df_copy = df_copy.sample(frac=1).reset_index(drop=True)
print(df_copy['size'].value_counts())
print("----------------------------------")
print(df_copy['shape'].value_counts())
print("----------------------------------")
print(df_copy['hemo_dist'].value_counts())
print("----------------------------------")
print(df_copy['inclusion'].value_counts())

NORM    5510
MICR     893
MACR     200
Name: size, dtype: int64
----------------------------------
NONE    4782
ECHI     570
OVAL     448
TEAR     347
SCHI     105
HELM      96
ACAN      95
ELLI      80
SPHE      41
SICK      30
BITE       9
Name: shape, dtype: int64
----------------------------------
NONE    5047
HYPO     817
HYPR     367
TARG     299
STOM      73
Name: hemo_dist, dtype: int64
----------------------------------
NONE                6074
RETI                 205
MALA                 121
HOJO                  74
BAST, RETI            56
PABO                  26
BAST                  22
NRBC                  18
BAST, HOJO, RETI       3
BAST, HOJO             2
HOJO, RETI             1
PABO, RETI             1
Name: inclusion, dtype: int64


In [77]:
shape_rare_list = ['ELLI','HELM','SPHE','SCHI','BITE','ACAN','SICK']             #no sickle cell

for item in shape_rare_list:
    shape_samples = np.where((df_copy['shape']==item))[0]
    len_samples = len(df_copy[df_copy['shape']==item])
    gap_num = min_samples - len_samples
    temp_df = df_copy.iloc[np.random.choice(shape_samples, size = gap_num)]
    df_copy = df_copy.append(temp_df, ignore_index = True)


df_copy = df_copy.sample(frac=1).reset_index(drop=True)
print(df_copy['size'].value_counts())
print("----------------------------------")
print(df_copy['shape'].value_counts())
print("----------------------------------")
print(df_copy['hemo_dist'].value_counts())
print("----------------------------------")
print(df_copy['inclusion'].value_counts())

NORM    6188
MICR    1159
MACR     200
Name: size, dtype: int64
----------------------------------
NONE    4782
ECHI     570
OVAL     448
TEAR     347
BITE     200
SPHE     200
HELM     200
SICK     200
ELLI     200
SCHI     200
ACAN     200
Name: shape, dtype: int64
----------------------------------
NONE    5978
HYPO     830
HYPR     367
TARG     299
STOM      73
Name: hemo_dist, dtype: int64
----------------------------------
NONE                7018
RETI                 205
MALA                 121
HOJO                  74
BAST, RETI            56
PABO                  26
BAST                  22
NRBC                  18
BAST, HOJO, RETI       3
BAST, HOJO             2
HOJO, RETI             1
PABO, RETI             1
Name: inclusion, dtype: int64


In [78]:
hemo_rare_list = ['STOM']

for item in hemo_rare_list:
    hemo_samples = np.where((df_copy['hemo_dist']==item))[0]
    len_samples = len(df_copy[df_copy['hemo_dist']==item])
    gap_num = min_samples - len_samples
    temp_df = df_copy.iloc[np.random.choice(hemo_samples, size = gap_num)]
    df_copy = df_copy.append(temp_df, ignore_index = True)


df_copy = df_copy.sample(frac=1).reset_index(drop=True)
print(df_copy['size'].value_counts())
print("----------------------------------")
print(df_copy['shape'].value_counts())
print("----------------------------------")
print(df_copy['hemo_dist'].value_counts())
print("----------------------------------")
print(df_copy['inclusion'].value_counts())

NORM    6310
MICR    1159
MACR     205
Name: size, dtype: int64
----------------------------------
NONE    4909
ECHI     570
OVAL     448
TEAR     347
BITE     200
SPHE     200
HELM     200
SICK     200
ELLI     200
SCHI     200
ACAN     200
Name: shape, dtype: int64
----------------------------------
NONE    5978
HYPO     830
HYPR     367
TARG     299
STOM     200
Name: hemo_dist, dtype: int64
----------------------------------
NONE                7144
RETI                 206
MALA                 121
HOJO                  74
BAST, RETI            56
PABO                  26
BAST                  22
NRBC                  18
BAST, HOJO, RETI       3
BAST, HOJO             2
HOJO, RETI             1
PABO, RETI             1
Name: inclusion, dtype: int64


In [79]:
inclusion_rare_list = ['RETI','MALA','HOJO','BAST, RETI','NRBC','PABO','BAST','BAST, HOJO','BAST, HOJO, RETI','HOJO, RETI','PABO, RETI']
min_samples = 400
for item in inclusion_rare_list:
    inclusion_samples = np.where((df_copy['inclusion']==item))[0]
    len_samples = len(df_copy[df_copy['inclusion']==item])
    gap_num = min_samples - len_samples
    temp_df = df_copy.iloc[np.random.choice(inclusion_samples, size = gap_num)]
    df_copy = df_copy.append(temp_df, ignore_index = True)


df_copy = df_copy.sample(frac=1).reset_index(drop=True)
print(df_copy['size'].value_counts())
print("----------------------------------")
print(df_copy['shape'].value_counts())
print("----------------------------------")
print(df_copy['hemo_dist'].value_counts())
print("----------------------------------")
print(df_copy['inclusion'].value_counts())


NORM    9854
MICR    1414
MACR     276
Name: size, dtype: int64
----------------------------------
NONE    8153
OVAL    1060
ECHI     582
TEAR     349
BITE     200
SPHE     200
HELM     200
SCHI     200
ACAN     200
SICK     200
ELLI     200
Name: shape, dtype: int64
----------------------------------
NONE    9672
HYPO     936
HYPR     432
TARG     299
STOM     205
Name: hemo_dist, dtype: int64
----------------------------------
NONE                7144
RETI                 400
HOJO, RETI           400
HOJO                 400
NRBC                 400
BAST                 400
PABO                 400
MALA                 400
BAST, HOJO           400
PABO, RETI           400
BAST, RETI           400
BAST, HOJO, RETI     400
Name: inclusion, dtype: int64


In [80]:
train_df_processed = df_copy
train_df_processed.to_csv('data/train_processed_new_clean1.csv',index=False)

In [81]:
processed_df = pd.read_csv('data/train_processed_new_clean1.csv')
processed_df

,image_name,size,shape,hemo_dist,inclusion
0,CDB_Sample030/jai_0000055_1047_43.png,NORM,NONE,NONE,NONE
1,EH5_Siemens012/jai_0001599_995_346.png,MICR,NONE,NONE,BAST
2,EH5_Siemens013/jai_0001064_585_247.png,NORM,TEAR,NONE,NONE
3,EH5_Siemens083/jai_0000185_1043_288.png,NORM,NONE,NONE,"BAST, RETI"
4,EH5_Siemens037/jai_0000135_1577_823.png,NORM,NONE,NONE,HOJO
...,...,...,...,...,...
11539,EH5_Siemens029/jai_0000654_363_644.png,NORM,HELM,NONE,NONE
11540,EH5_Siemens041/jai_0000231_872_1284.png,MICR,SCHI,NONE,NONE
11541,CDB_Sample012/jai_0000066_74_797.png,NORM,NONE,STOM,NONE
11542,EH5_Siemens083/jai_0001472_880_1111.png,MACR,NONE,HYPR,NONE


In [82]:
print(test_df['size'].value_counts())
print("----------------------------------")
print(test_df['shape'].value_counts())
print("----------------------------------")
print(test_df['hemo_dist'].value_counts())
print("----------------------------------")
print(test_df['inclusion'].value_counts())

NORM    1927
MICR     226
MACR      30
Name: size, dtype: int64
----------------------------------
NONE    1719
ECHI     126
OVAL     125
TEAR      76
SCHI      35
HELM      30
ACAN      25
ELLI      25
SPHE      13
SICK       7
BITE       2
Name: shape, dtype: int64
----------------------------------
NONE    1822
HYPO     186
TARG      78
HYPR      74
STOM      23
Name: hemo_dist, dtype: int64
----------------------------------
NONE          2053
RETI            54
MALA            30
HOJO            13
BAST            11
BAST, RETI      10
NRBC             5
PABO             4
HOJO, RETI       2
BAST, HOJO       1
Name: inclusion, dtype: int64
